[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/geraldmc/irilab2026/blob/main/notebooks/r2/r2-q1/01_pv_orientation.ipynb)

# R2-Q1 Week 1 — Orient on PlantVillage

This notebook orients you on the PlantVillage dataset: the 38 disease classes, the lab-condition image structure, and the per-class image distribution. Together with `02_pd_orientation.ipynb`, the inspection here supports your Week 1 EQ#1 report and the written pre-commits on aggregation level, class-space remapping, and statistical test that R2-Q1 requires before any training begins.

By the end of this notebook you will have:

- A PlantVillage metadata table (`pv_metadata.parquet`) keyed by image path, with class label and host species per image, ready for the classifier in Week 2.
- Summary statistics on the PlantVillage class space — total image count, per-class distribution, host-species coverage — saved as `pv_class_summary.parquet`.
- A `pv_orientation_summary.json` recording dataset path, image and class counts, host-species count, and key distributional facts, ready as Week 1 EQ#1 input.

In [ ]:
try:
    import irilab2026 as iri
    print(f"OK — irilab2026 {iri.__version__} is installed and all deps importable.")
    print("    → use Pattern 2 in Cell 1:")
    print("      !pip install --force-reinstall --no-deps git+https://github.com/geraldmc/irilab2026.git@main")
except ModuleNotFoundError as e:
    print(f"MISSING — {e.name} not importable.")
    print("    → use Pattern 1 in Cell 1:")
    print("      !pip install git+https://github.com/geraldmc/irilab2026.git@main")

In [ ]:
# Pattern 1 — fresh or partial runtime (installs deps that aren't present yet)
# This skips anything pip already sees as installed. This is ideal for a new environment or when you want 
# to be sure everything is up to date.

# !pip install git+https://github.com/geraldmc/irilab2026.git@main

# Pattern 2 — populated runtime (refreshes irilab2026 only, leaves deps alone)
# This is ideal for when you already have the dependencies installed and just want to update irilab2026.

# !pip install --force-reinstall --no-deps git+https://github.com/geraldmc/irilab2026.git@main

**A note about a possible restart prompt.** After running the install
cell above, Colab may show a banner that says **"RESTART SESSION"** with
a button to click. This happens because the install can bring in newer
versions of some libraries that the running Python session is already
using.

If you see the banner:

1. Click **RESTART SESSION** (or use **Runtime → Restart session**).
2. Wait a few seconds for the kernel to reconnect.
3. Run the next cell. The packages from the install are already on
   disk, so the install cell does not need to run again.

If you don't see the banner, ignore this note and move on. The restart
is only needed once per Colab session.

**A note about an `HF_TOKEN` warning.** When you run the cell that loads
PlantVillage in Section 2, you may see a warning that mentions setting
an `HF_TOKEN` in your Colab secrets. You can ignore it.

The warning is Hugging Face — the service that hosts the image data —
suggesting that you log in for slightly faster downloads. Public
datasets like this one work without logging in, and the download speed
without a token is already fine for this notebook.

In [ ]:
# Mount Drive, set up irilab2026, point to the R2-Q1 output directory.
import irilab2026 as iri
iri.setup()

OUTPUT_DIR = iri.output_dir("r2_q1")
print(f"Output directory: {OUTPUT_DIR}")

## 2) Load the PlantVillage metadata

PlantVillage is the dataset of plant disease photographs you'll be training
your classifier on next week. It contains roughly 54,000 images across 38
classes, where each class is a unique combination of a plant species and a
disease (or "healthy"). Every image is a single leaf, photographed on a
plain background under controlled lighting.

The function below returns **two things** at once: a pandas DataFrame
describing every image (host species, disease, train/test assignment, and
so on — one row per image) and a separate image dataset you can index
into to fetch the pixels for any row. You'll spend most of this notebook
looking at the DataFrame; the images come into the picture in Section 6.

The first time you run this in a fresh session, expect about a minute as
roughly 855 MB of image data downloads to your Drive cache. Subsequent
sessions reuse the cached data, so the download only happens once.

In [ ]:
### 2.1) Load PlantVillage metadata
metadata, images = iri.load_plantvillage()

print(f"Metadata shape:  {metadata.shape[0]:,} rows × {metadata.shape[1]} columns")
print(f"Columns:         {list(metadata.columns)}")
print(f"Image dataset:   {len(images):,} images")
print()
print("First five rows of metadata:")
metadata.head()

Seven columns. Two of them — `class_label` and `class_idx` — carry the
same information in different forms. `class_label` is the human-readable
string ("Tomato___Tomato_mosaic_virus"); `class_idx` is an integer (0 to
37) assigned by sorting class labels alphabetically. The integer form is
what your classifier will predict next week.

The next three — `host`, `disease`, `split` — are the columns you'll
work with most. `host` and `disease` are the two pieces of `class_label`
pulled apart for you. `split` is the train-or-test assignment you'll use
when you train the classifier in Notebook 03.

The last two — `leaf_id` and `leaf_grouped` — come up in Notebook 03
when you set up cross-validation. You can ignore them for now.

The image dataset is a separate object. Each row of the metadata
DataFrame corresponds to one image in the image dataset, accessible by
the row's index. To pull the image for row `i` from the metadata:

    img = images[i]["image"]

This pattern — DataFrame for the descriptions, image dataset for the
pixels — runs through the rest of the R2-Q1 notebooks. You'll see it in
action in Section 6.

## 3) Inspect class structure

PlantVillage organizes images into classes by folder name on disk. The
folder name encodes both the plant species and the disease (or "healthy")
in a single string, with three underscores separating the two parts.
This convention is worth understanding directly, because the parsing
choices made by `load_plantvillage()` are what make `host` and `disease`
available as separate columns.

Start by looking at one row in detail.

In [ ]:
### 3.1) Walk through one row
example = metadata.iloc[0]
print(f"class_label:   {example['class_label']}")
print(f"class_idx:     {example['class_idx']}")
print(f"host:          {example['host']}")
print(f"disease:       {example['disease']}")
print(f"split:         {example['split']}")
print(f"leaf_id:       {example['leaf_id']}")
print(f"leaf_grouped:  {example['leaf_grouped']}")
print()

# The matching image. Note the lookup pattern: int(example.name) gives the
# row's position in the image dataset.
example_image = images[int(example.name)]["image"]
print(f"image:         PIL Image, {example_image.size[0]}×{example_image.size[1]} pixels, mode={example_image.mode}")

The class label `Tomato___Tomato_mosaic_virus` is the literal folder name
on disk. The loader splits it on the triple underscore (`___`),
takes the left side as host and the right side as disease, then
replaces remaining underscores with spaces in both. Healthy classes work
the same way: `Apple___healthy` becomes host `Apple`, disease `healthy`.

Two conventions worth noting now, because they'll matter when you write
your class-space remapping in Notebook 02:

- Diseases like "Bacterial spot" appear on multiple hosts (tomato,
  pepper, peach). In the class space, these are *different classes*
  ("Tomato___Bacterial_spot" vs "Pepper,_bell___Bacterial_spot"), even
  though the disease is the same.
- "healthy" is a disease label too, applied to multiple hosts. A
  classifier trained on these 38 classes is technically learning
  "host plus disease," not just "disease."

Now count the classes and look at how images are distributed across them.

In [ ]:
### 3.2) Count classes and per-class images
class_counts = metadata['class_label'].value_counts()

print(f"Number of classes:  {len(class_counts)}")
print()
print(f"Largest class:      {class_counts.idxmax()}")
print(f"                    {class_counts.max():,} images")
print(f"Smallest class:     {class_counts.idxmin()}")
print(f"                    {class_counts.min():,} images")
print(f"Median count:       {int(class_counts.median()):,} images per class")
print(f"Ratio largest:smallest:  {class_counts.max() / class_counts.min():.1f}×")

Notice the ratio between the largest and smallest classes. PlantVillage
is not balanced — the largest class has many times more images than the
smallest. This matters for two reasons.

When you train a classifier next week, the model sees the most-represented
classes far more often than the least-represented ones during training,
which biases its predictions toward the larger classes unless you do
something about it. Common somethings include weighted sampling and
class-weighted loss; we'll come back to this in Notebook 03.

When you measure transfer accuracy in Notebook 04, per-class accuracy
on the smallest classes will be calculated from very few test images.
The Considerations section on the R2-Q1 page calls this out as one of
the reasons to aggregate to coarser groupings (host or disease type)
before reporting accuracy. The imbalance you're seeing here is part of
why that matters.

In [ ]:
### 3.3) Plot per-class image counts
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

# -----------------------------
# Prepare dataframe
# -----------------------------
plot_df = (
    class_counts
    .sort_values(ascending=False)   # descending order
    .reset_index()
)

plot_df.columns = ["class", "count"]

# -----------------------------
# Categorize counts
# -----------------------------
def count_category(n):
    if n > 3000:
        return "> 3000"
    elif n > 2000:
        return "2001–3000"
    elif n > 1000:
        return "1001–2000"
    else:
        return "≤ 1000"

plot_df["count_group"] = plot_df["count"].apply(count_category)

group_order = ["> 3000", "2001–3000", "1001–2000", "≤ 1000"]

palette = {
    "> 3000": "#9ecae1",
    "2001–3000": "#a1d99b",
    "1001–2000": "#fdae6b",
    "≤ 1000": "#fbb4b9",
}

# -----------------------------
# Plot
# -----------------------------
fig, ax = plt.subplots(figsize=(10, 10))

sns.barplot(
    data=plot_df,
    y="class",
    x="count",
    hue="count_group",
    hue_order=group_order,
    palette=palette,
    dodge=False,
    order=plot_df["class"],   # preserve descending order
    ax=ax
)

ax.set_xlabel("Number of images")
ax.set_ylabel("")
ax.set_title("Images per class in PlantVillage")

# Move legend higher on the right
ax.legend(
    title="Image count",
    loc="center right",
    bbox_to_anchor=(0.98, 0.78),
    frameon=True
)

sns.despine(left=True, bottom=False)

plt.tight_layout()
plt.show()

## 4) Inspect host species

"Host" here means the plant species the leaf came from. Aggregating from
classes to hosts collapses across diseases — every disease of tomato,
plus healthy tomato, becomes one "Tomato" group. This is one of the
candidate aggregation levels for your transfer evaluation: rather than
asking "how does my classifier do on tomato bacterial spot?" you ask
"how does it do on tomatoes overall?"

In [ ]:
### 4.1) Count hosts and images per host
host_counts = metadata['host'].value_counts()

print(f"Number of host species:  {len(host_counts)}")
print()
print("Images per host:")
print(host_counts.to_string())

A handful of hosts dominate PlantVillage by image count (tomato in
particular). Several hosts appear in only one class — they're represented
by their healthy-only images, with no diseased examples of that host in
the dataset at all. Blueberry, raspberry, and soybean are the cases to
watch for.

This has a direct consequence for what you can measure. A classifier
trained on PV that's then asked to identify a diseased blueberry leaf
in PlantDoc has no PV examples of diseased blueberry to have learned
from. Whatever it predicts will be guesswork from healthy-blueberry
features plus disease features learned from other hosts. When you see
unexpectedly poor transfer on certain hosts in Notebook 04, this is
where to look first.

In [ ]:
### 4.2) Plot images per host
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

# -----------------------------
# Prepare dataframe
# -----------------------------
plot_df = (
    host_counts
    .sort_values(ascending=False)
    .reset_index()
)

plot_df.columns = ["host_species", "count"]

# -----------------------------
# Categorize counts
# Adjust thresholds as needed
# -----------------------------
def count_category(n):
    if n > 3000:
        return "> 3000"
    elif n > 2000:
        return "2001–3000"
    elif n > 1000:
        return "1001–2000"
    else:
        return "≤ 1000"

plot_df["count_group"] = plot_df["count"].apply(count_category)

group_order = ["> 3000", "2001–3000", "1001–2000", "≤ 1000"]

palette = {
    "> 3000": "#9ecae1",
    "2001–3000": "#a1d99b",
    "1001–2000": "#fdae6b",
    "≤ 1000": "#fbb4b9",
}

# -----------------------------
# Plot
# -----------------------------
fig, ax = plt.subplots(figsize=(9, 5))

sns.barplot(
    data=plot_df,
    x="host_species",
    y="count",
    hue="count_group",
    hue_order=group_order,
    palette=palette,
    dodge=False,
    order=plot_df["host_species"],
    ax=ax
)

ax.set_xlabel("")
ax.set_ylabel("Number of images")
ax.set_title("Images per host species")

plt.xticks(rotation=45, ha="right")

ax.legend(
    title="Image count",
    loc="upper right",
    frameon=True
)

sns.despine(left=False, bottom=False)

plt.tight_layout()
plt.show()

## 5) Inspect disease types

The `disease` column carries a different cut through the same data. Where
the class label tells you "tomato with bacterial spot" and the host
column tells you "tomato," the disease column tells you "bacterial spot"
regardless of which host it appeared on. Aggregating from classes to
diseases collapses across hosts.

Start with the simplest cut: healthy versus diseased.

In [ ]:
### 5.1) Healthy vs diseased
is_healthy = metadata['disease'].str.lower().str.contains('healthy')

n_healthy = is_healthy.sum()
n_diseased = (~is_healthy).sum()
print(f"Healthy images:   {n_healthy:,}  ({n_healthy / len(metadata):.1%})")
print(f"Diseased images:  {n_diseased:,}  ({n_diseased / len(metadata):.1%})")

Now look at how many distinct disease labels there are, and what they
are.

In [ ]:
### 5.2) Count and list distinct diseases
unique_diseases = sorted(metadata['disease'].unique())

print(f"Number of distinct disease labels:  {len(unique_diseases)}")
print()
print("All disease labels:")
for d in unique_diseases:
    print(f"  {d}")

The R2-Q1 Considerations section flagged "fungal, bacterial, or viral
disease categories" as one of the candidate aggregation levels for your
transfer evaluation. To use that grouping, you need a mapping from each
disease label to its broad category.

PlantVillage does not provide this mapping. Somebody has to write it.
The cell below provides a starter mapping covering the disease labels
in the dataset, based on common plant-pathology references. **Treat it
as a draft, not as authoritative.** Each entry is a verifiable claim
("Late blight is fungal") that you should check against your reading
list before locking it in as part of your Notebook 02 pre-commit.

One disease in PlantVillage doesn't fit cleanly into the three
categories: spider mites are an arthropod pest, not a disease in the
pathological sense. That's a real classification choice you'll need to
make in Notebook 02 — group it with the others under some "other"
category, drop it from your analysis, or treat it on its own.

In [ ]:
### 5.3) Apply a starter category mapping
# Starter mapping from disease label to broad category.
# Categories: fungal, bacterial, viral, pest, healthy.
# Verify each entry against your reading list before using this in
# Notebook 02. Disease names below match what `load_plantvillage()`
# produces in the `disease` column.
disease_category = {
    "Apple scab":                                "fungal",
    "Black rot":                                 "fungal",
    "Cedar apple rust":                          "fungal",
    "Powdery mildew":                            "fungal",
    "Cercospora leaf spot Gray leaf spot":       "fungal",
    "Common rust":                               "fungal",
    "Northern Leaf Blight":                      "fungal",
    "Esca (Black Measles)":                      "fungal",
    "Leaf blight (Isariopsis Leaf Spot)":        "fungal",
    "Early blight":                              "fungal",
    "Late blight":                               "fungal",
    "Leaf Mold":                                 "fungal",
    "Septoria leaf spot":                        "fungal",
    "Target Spot":                               "fungal",
    "Leaf scorch":                               "fungal",
    "Bacterial spot":                            "bacterial",
    "Haunglongbing (Citrus greening)":           "bacterial",
    "Tomato mosaic virus":                       "viral",
    "Tomato Yellow Leaf Curl Virus":             "viral",
    "Spider mites Two-spotted spider mite":      "pest",
    "healthy":                                   "healthy",
}

# Apply the mapping. Any disease label not in the dict shows up as "UNMAPPED" —
# if you see that, your mapping is incomplete.
categorized = metadata['disease'].map(disease_category).fillna("UNMAPPED")

print("Images per category:")
print(categorized.value_counts().to_string())
print()
unmapped_count = (categorized == "UNMAPPED").sum()
if unmapped_count > 0:
    unmapped_diseases = metadata.loc[categorized == "UNMAPPED", 'disease'].unique()
    print(f"WARNING: {unmapped_count} images have unmapped disease labels:")
    for d in unmapped_diseases:
        print(f"  {d}")

A few observations to carry forward.

The category counts are even more imbalanced than the per-class counts.
Fungal diseases dominate; bacterial and viral are minority categories.
If you aggregate to the three-category level for your transfer evaluation,
you'll have only a few hundred test images per category in PlantDoc —
small but still workable, which is the point of aggregating.

The "pest" category contains exactly one disease label (spider mites)
on exactly one host (tomato). Whether to include it depends on your
research question framing. If "fungal vs bacterial vs viral" is the
comparison you care about, drop it. If "controlled pathology vs
real-world" is the comparison, keep it.

The "healthy" category is structurally different from the others — it's
not a disease, it's the absence of one. Most published lab/field gap
analyses report performance on the disease-only subset, treating healthy
as a separate concern. Decide how you'll handle it in Notebook 02.

## 6) Look at sample images

You've inspected the metadata. Now look at the actual images, because the
metadata describes what's *in* PlantVillage, not what its photographs
*look like*. The visual character of the dataset — the staging, the
lighting, the framing — is what makes the lab-to-field gap exist in the
first place.

The R2-Q1 Considerations section names three properties of PV imagery
that contribute to the gap: solid backgrounds, controlled lighting, and
single-leaf framing. Each of those is something you can see directly
in the next cell.

The display cell pulls images from one healthy and one diseased example
of four representative host species. The choice of which exact images
to show is reproducible — running this cell again gets you the same
images, so you can refer back to specific ones in your Notebook 02
write-up.

In [ ]:
### 6.1) Display sample images

# Four hosts chosen to span the dataset: a fruit tree (apple), a vine
# (grape), a tuber (potato), and the most-represented host (tomato).
# Each contributes one healthy and one diseased sample.
sample_hosts = ['Apple', 'Grape', 'Potato', 'Tomato']

samples = []
for host in sample_hosts:
    for state, mask in [('healthy', is_healthy), ('diseased', ~is_healthy)]:
        host_rows = metadata[(metadata['host'] == host) & mask]
        if len(host_rows) == 0:
            continue
        # Fixed seed for reproducibility.
        chosen = host_rows.sample(n=1, random_state=42).iloc[0]
        samples.append((state, chosen))

n = len(samples)
fig, axes = plt.subplots(n // 2, 2, figsize=(8, 4 * (n // 2)))
for ax, (state, row) in zip(axes.flat, samples):
    # Look up the image in the image dataset by the row's original index.
    img = images[int(row.name)]["image"]
    ax.imshow(img)
    ax.set_title(f"{row['host']} — {row['disease']}", fontsize=11)
    ax.axis('off')
plt.tight_layout()
plt.show()

A few things to take in.

**The background is uniform.** Every image is shot on a plain backdrop —
gray, sometimes off-white. There is no soil, no other plants, no
laboratory equipment, no human hands. A model trained on these images
has every opportunity to learn that "uniform gray" is part of what a
plant disease photograph looks like. When that model later sees a
field photograph with a cluttered background, the cue it learned no
longer applies.

**The lighting is consistent.** Shadows are minimal and directional in a
predictable way. Exposure is roughly the same across images. Compare
this to a field photograph where the sun moves, the camera angle
varies, and parts of the leaf are in shadow.

**One leaf per image, centered and flat.** PlantVillage photographers
removed leaves from plants and arranged them on the backdrop before
shooting. A field photograph shows leaves still attached to the plant,
at the angle the plant grew them, possibly partially occluded by other
leaves.

None of this is a problem for PlantVillage as a dataset — it's exactly
what makes it useful as a controlled-condition reference. The problem
arises when a model trained on these images is evaluated on a different
kind of image without an explicit account of what changed. The next
notebook (`02_pd_orientation`) shows you what the other kind of image
looks like.

## 7) What's next

You've now seen PlantVillage's metadata structure (38 classes, the host ×
disease encoding), its imbalances (across classes, across hosts, across
disease categories), and its visual character (controlled-condition
photographs of single leaves).

Notebook 02 (`02_pd_orientation.ipynb`) does the same orientation work
for **PlantDoc**, the field-photograph dataset you'll evaluate against
in Week 3. Notebook 02 also produces your Week 1 EQ#1 deliverable —
a `precommit.json` capturing three decisions:

- Your **aggregation level** for the transfer evaluation (host group,
  disease category, or another grouping you defend).
- Your **class-space remapping** from PlantVillage's 38 classes to
  PlantDoc's class space at your chosen aggregation level.
- Your **statistical test** for whether the PV-internal accuracy and
  PV → PD accuracy are distinguishable.

The reason all three commitments live in Notebook 02 rather than here
is that you need to have seen PlantDoc's class space before you can
write a sensible remapping. PlantDoc has 27 classes, structured
differently from PV's 38 — what looks like a natural mapping from
PV-side alone may not survive contact with PD-side.

This notebook produced no saved files. Move on to Notebook 02 when
ready.